# Spark preprocessing and replay

Measures local or standalone Spark aggregation, feature extraction, and replay throughput. A second worker is optional; configure addresses locally using ../docs/REPRODUCING.md.

Run the setup cell first. Full processing and training require locally provisioned datasets; the quick start uses synthetic tests and saved aggregate results.


In [ ]:
# Load the libraries and project settings needed for this step.
from pathlib import Path
import sys, json, os, time, shutil
# Find the project from its configuration file rather than a machine-specific path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
# Create the destination directory before writing any outputs.
from IPython.display import display
from hemodynamics import cluster as C, spark_jobs as J, figstyle as F
# Create the destination directory before writing any outputs.
cfg = C.load_config(ROOT)
FIG = ROOT / 'figures' / 'spark'; FIG.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('This machine (address Spark binds to):', C.lan_ipv4(), '| master: spark://%s:%s' % (C.master_host(cfg), cfg['master_port']))
print('Java for Spark:', C.find_jdk(cfg))
print(f'Free disk: {shutil.disk_usage(ROOT).free / 1e9:.0f} GB')

## 1. Start (or attach to) the cluster and look at it
`ATTACH_ONLY = True` never starts anything and just attaches to a master that already runs (use it on the real two-machine cluster: master and desktop worker started with the scripts, laptop worker started by the kit).

In [ ]:
ATTACH_ONLY = False
if ATTACH_ONLY:
    workers = C.alive_workers(C.master_view(C.master_host(cfg), cfg['master_ui']))
else:
    workers = C.cluster_up(ROOT)      # master + 'desktop' (8 cores, 24 GB) + 'sim2' (4 cores, 6 GB); skips whatever already runs
display(pd.DataFrame([{'worker id': w['id'], 'host': w['host'], 'cores': w['cores'], 'memory (MB)': w['memory'], 'state': w['state']} for w in workers]))
print(C.run_script('status.ps1', root=ROOT))

In [ ]:
# Check the saved outputs before continuing to the next stage.
spark = C.get_spark('etl', root=ROOT, streaming=True)
rep = C.cluster_report(spark)
C.print_report(rep)
checks = {'at least two workers registered': len(rep['workers_registered']) >= 2,
          'tasks ran on every registered worker': len(rep['workers_running_tasks']) >= len(rep['workers_registered']),
          'no Python / PySpark / torch / code-hash mismatch between workers': not rep['mismatches']}
for k, v in checks.items(): print(('PASS  ' if v else 'FAIL  ') + k)
print('Spark UIs: master', spark._bshi['master_ui'], '| application', spark._bshi['driver_ui'])

## 2. Smoke job: Spark aggregates the dev subset, pandas checks it
Each task reads its parquet parts from **its own machine's** project root with pyarrow (a laptop needs its own copy of the data shard, not the desktop's path). The table must equal pandas exactly.

In [ ]:
# Collect the per-record values into a table for comparison.
sm = J.smoke_aggregate(spark, ROOT)
display(sm['table'].round(3))
display(pd.DataFrame(sm['tasks_by_worker']))
print('executors seen by the Spark UI (REST):', sm['executors'])
for k, v in {'counts equal to pandas (17,167 windows)': sm['counts_equal'] and sm['windows'] == 17167, 'weight sums equal to pandas': sm['weights_equal'],
             'tasks ran on both workers': len(sm['tasks_by_worker']) >= 2}.items(): print(('PASS  ' if v else 'FAIL  ') + k)

## 3. B1: distributed window-feature extraction (1 worker vs all workers)
One Spark partition per dev-subset file; each task decodes its windows and computes 34 features (25 context values, ECG/ART/PPG plane mean and SD over the valid beats, valid-beat counts). The features of 100 random windows are compared with a **fresh** `bshi.assemble` of the same windows.

In [ ]:
rows = []
for label, kw in (('desktop worker only (8 cores)', {'workers': 1}), ('desktop + second worker (12 cores)', {})):
    s = C.get_spark('etl', root=ROOT, streaming=True, **kw)
    J.run_features(s, ROOT)                                  # warm-up
    pdf, tm = J.run_features(s, ROOT)
    rows.append({'configuration': label, 'executors': tm['executors'], 'windows': tm['windows'], 'seconds': tm['seconds'], 'windows_per_s': tm['windows_per_s'], 'windows_by_worker': tm['windows_by_worker']})
# Collect the per-record values into a table for comparison.
b1 = pd.DataFrame(rows); display(b1)
ver = J.verify_features(ROOT, pdf)
print(ver)
# Create the plotting panels for this comparison.
print('PASS  context features equal a fresh assembly' if ver['context_exact'] else 'FAIL  context features differ')
print('PASS  plane statistics within float16 storage error' if ver['plane_stats_within_float16'] else 'FAIL  plane statistics differ')
print('speed-up with the second worker: %.2fx (simulated: both workers share this PC)' % (b1.windows_per_s.iloc[1] / b1.windows_per_s.iloc[0]))
# Label the panel with the measurement names and their units.
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
# Save the labeled figure for the analysis report.
ax[0].bar(range(len(b1)), b1.windows_per_s, color=['#1f77b4', '#2ca02c'])
ax[0].set_xticks(range(len(b1))); ax[0].set_xticklabels(['desktop only', 'desktop + sim2'])
# Save the labeled figure for the analysis report.
F.finish_axes(ax[0], 'cluster configuration (simulated: workers share one CPU)', 'feature extraction speed (windows/s)', 'B1 speed')
byw = b1.windows_by_worker.iloc[1]
ax[1].bar(list(byw), list(byw.values()), color='#2ca02c')
F.finish_axes(ax[1], 'worker', 'windows processed (count)', 'B1 work split with both workers')
fig.tight_layout(); F.audit_figure(fig, 'fig_b1_scaling'); fig.savefig(FIG / 'fig_b1_scaling.png', dpi=90); plt.show()

## 4. B2: Structured Streaming replay
Test-split windows are replayed at a fixed rate, 4 patients at a time, each patient in chronological order. A `rate` source is the clock (no shared folder is needed, so the laptop can take part); each micro-batch looks its windows up in the worker's own data, computes the features and a placeholder risk score (`default_score`; the trained model replaces it next queue). Latency = from the scheduled emission of a window until its result is back on the driver.

In [ ]:
# Inspect the selected settings and outputs for this notebook step.
spark = C.get_spark('etl', root=ROOT, streaming=True)
rows = []
for rate in (25, 50, 100):
    st, res = J.stream_replay(spark, ROOT, n_windows=600, rate=rate)
    rows.append(st)
# Collect the per-record values into a table for comparison.
b2 = pd.DataFrame(rows)
display(b2[['rate_windows_per_s', 'windows', 'latency_ms_p50', 'latency_ms_p95', 'latency_ms_p99', 'throughput_windows_per_s', 'micro_batches', 'batch_size_median', 'windows_by_worker', 'chronological_within_patient']].round(1))
print('PASS  every replayed window came back' if (b2.windows == 600).all() else 'FAIL  some windows are missing')
print('PASS  per-patient chronological order kept' if b2.chronological_within_patient.all() else 'FAIL  order broken')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for q, c in (('latency_ms_p50', '#2ca02c'), ('latency_ms_p95', '#ff7f0e'), ('latency_ms_p99', '#d62728')):
    ax[0].plot(b2.rate_windows_per_s, b2[q], marker='o', color=c, label=q.replace('latency_ms_', ''))
# Save the labeled figure for the analysis report.
ax[0].legend(title='percentile', fontsize=8)
F.finish_axes(ax[0], 'offered replay rate (windows/s)', 'end-to-end latency (ms)', 'B2 latency')
ax[1].plot(b2.rate_windows_per_s, b2.throughput_windows_per_s, marker='o', color='#1f77b4', label='measured')
ax[1].plot(b2.rate_windows_per_s, b2.rate_windows_per_s, ls='--', color='grey', label='offered')
ax[1].legend(fontsize=8)
F.finish_axes(ax[1], 'offered replay rate (windows/s)', 'windows scored (windows/s)', 'B2 throughput')
fig.tight_layout(); F.audit_figure(fig, 'fig_b2_stream'); fig.savefig(FIG / 'fig_b2_stream.png', dpi=90); plt.show()

## 5. B3: trial harness (used for the optimizer / learning-rate sweeps and the ablations)
The `gpu` profile gives every machine **one** task slot (`spark.executor.cores = spark.task.cpus`), so a trial owns that machine and its GPU. `run_trials` returns metrics through Spark (no shared files); a failed trial is retried once (possibly on the other machine) and every failure stays in the report. Here: cheap demo trials with planned failures, then one real GPU trial per machine.

In [ ]:
# Collect the per-record values into a table for comparison.
spark = C.get_spark('gpu', root=ROOT)
rep_gpu = C.cluster_report(spark)
# Collect the per-record values into a table for comparison.
print('profile gpu: executor cores', rep_gpu['executor_cores'], '= task cpus', rep_gpu['task_cpus'], '| tasks ran on', sorted(rep_gpu['workers_running_tasks']))
cfgs = [{'lr': 1e-3, 'sleep': 1}, {'lr': 3e-3, 'sleep': 1}, {'lr': 1e-2, 'sleep': 1, 'fail_on_attempt': 0}, {'lr': 1e-1, 'sleep': 1, 'fail_on_attempt': 0}, {'lr': 5e-4, 'sleep': 1}, {'lr': 2e-3, 'sleep': 1}]
trials = C.run_trials(spark, cfgs, retries=1)
# Collect the per-record values into a table for comparison.
display(pd.DataFrame(trials)[['trial_id', 'ok', 'attempts', 'worker', 'seconds', 'metrics', 'error']])
print('PASS  all trials ended ok' if all(t['ok'] for t in trials) else 'FAIL  a trial failed')
# Collect the per-record values into a table for comparison.
print('PASS  the two planned failures were retried and reported' if [t['attempts'] for t in trials] == [1, 1, 2, 2, 1, 1] and trials[2]['history'][0]['error'] else 'FAIL  retry logic')
gpu = C.run_trials(spark, [{'n': 4096, 'iters': 20}] * 2, fn='hemodynamics.cluster:gpu_trial')
display(pd.DataFrame(gpu)[['trial_id', 'ok', 'worker', 'seconds', 'metrics', 'error']])

## 6. After the laptop joined
Start the laptop worker with the kit (`docs/REPRODUCING.md`), then rerun this cell. It repeats the B1 comparison and the GPU trial with the **real laptop** and writes `results/spark_cluster_laptop.json`; until a worker named `laptop` is registered it only says so.

In [ ]:
# Run the configured processing step and retain its returned summary.
view = C.master_view(C.master_host(cfg), cfg['master_ui'])
s = C.get_spark('etl', root=ROOT, streaming=True)
rep_now = C.cluster_report(s)
if 'laptop' not in rep_now['workers_running_tasks']:
    print('The laptop worker is not registered yet (workers running tasks:', sorted(rep_now['workers_running_tasks']), '). Start it with the kit and rerun this cell.')
else:
    C.print_report(rep_now)
    out = {'report': rep_now, 'b1': []}
    for label, kw in (('desktop only', {'workers': 1}), ('desktop + laptop', {})):
        s = C.get_spark('etl', root=ROOT, streaming=True, **kw)
        J.run_features(s, ROOT); pdf, tm = J.run_features(s, ROOT)
        out['b1'].append({'configuration': label, **tm})
    g = C.get_spark('gpu', root=ROOT)
    out['gpu_trials'] = C.run_trials(g, [{'n': 4096, 'iters': 20}] * 2, fn='hemodynamics.cluster:gpu_trial')
    display(pd.DataFrame(out['b1']))
    display(pd.DataFrame(out['gpu_trials'])[['trial_id', 'ok', 'worker', 'metrics', 'error']])
    (ROOT / 'results' / 'spark_cluster_laptop.json').write_text(json.dumps(out, indent=1, default=str), encoding='utf-8')

## 7. Save the evidence and stop the Spark session
The master and the workers keep running (stop them with `scripts/spark/stop_all.ps1`, or run the last line of the cell).

In [ ]:
# Store the settings and results as JSON for later inspection.
ev = {'mode': 'simulated: second worker sim2 on the same PC' if 'laptop' not in rep_now['workers_running_tasks'] else 'real laptop joined', 'cluster_report': rep, 'gpu_report': rep_gpu,
      'smoke': {k: v for k, v in sm.items() if k != 'table'}, 'b1': b1.to_dict('records'), 'b1_verify': ver, 'b2': b2.to_dict('records'),
      'b3_trials': [{k: t[k] for k in ('trial_id', 'ok', 'attempts', 'worker', 'seconds', 'metrics', 'error')} for t in trials], 'b3_gpu': [{k: t[k] for k in ('worker', 'metrics', 'error')} for t in gpu],
      'windows_os_notes': ['no winutils: Spark never writes local files', 'executors import hemodynamics from PYTHONPATH on their own machine', 'pysite/sitecustomize.py flushes the PySpark 4.0.1 worker socket', 'streaming checkpoints via compiled NoPermLocalFileSystem (driver only)']}
(ROOT / 'results' / 'spark_cluster_verification.json').write_text(json.dumps(ev, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o)), encoding='utf-8')
print('saved results/spark_cluster_verification.json')
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active is not None: active.stop()
# C.cluster_down(ROOT)   # stop the master and the workers as well